# 04 — Aggregations & GroupBy

**What you will learn:**
- `groupBy().agg()` — the core aggregation pattern
- Built-in aggregate functions: count, sum, avg, min, max, stddev
- Multiple aggregations in one pass
- `countDistinct()`, `collect_list()`, `collect_set()`
- `pivot()` — reshape rows into columns
- `rollup()` and `cube()` — hierarchical subtotals
- `agg()` without `groupBy` — whole-DataFrame aggregation

## Setup

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import (
    col, count, countDistinct, sum, avg, min, max,
    stddev, variance, round,
    collect_list, collect_set,
    first, last
)

schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("location",   StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("year",       IntegerType(), True),
])

data = [
    (1,  "Alice",   "Engineering", "NYC",    95000.0,  2023),
    (2,  "Bob",     "Marketing",   "NYC",    72000.0,  2023),
    (3,  "Charlie", "Engineering", "SF",     105000.0, 2023),
    (4,  "Diana",   "HR",          "NYC",    68000.0,  2023),
    (5,  "Eve",     "Marketing",   "SF",     78000.0,  2023),
    (6,  "Frank",   "Engineering", "SF",     88000.0,  2022),
    (7,  "Grace",   "HR",          "NYC",    71000.0,  2022),
    (8,  "Hank",    "Engineering", "NYC",    92000.0,  2022),
    (9,  "Ivy",     "Marketing",   "SF",     75000.0,  2022),
    (10, "Jake",    "Engineering", "NYC",    110000.0, 2022),
]

df = spark.createDataFrame(data, schema=schema)
df.show()

## 1. count() — Row Count per Group

In [ ]:
df.groupBy("department").count().show()

## 2. sum(), avg(), min(), max()

In [ ]:
df.groupBy("department").agg(
    sum("salary").alias("total_salary"),
    avg("salary").alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary")
).show()

## 3. Multiple Aggregations — All in One agg()

In [ ]:
from pyspark.sql.functions import round

df.groupBy("department").agg(
    count("emp_id").alias("headcount"),
    countDistinct("location").alias("num_locations"),
    round(avg("salary"), 2).alias("avg_salary"),
    sum("salary").alias("total_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    round(stddev("salary"), 2).alias("stddev_salary")
).orderBy("department").show()

## 4. Group by Multiple Columns

In [ ]:
df.groupBy("department", "location").agg(
    count("emp_id").alias("headcount"),
    round(avg("salary"), 2).alias("avg_salary")
).orderBy("department", "location").show()

## 5. Whole-DataFrame Aggregation (no groupBy)

In [ ]:
df.agg(
    count("emp_id").alias("total_employees"),
    round(avg("salary"), 2).alias("company_avg_salary"),
    sum("salary").alias("total_payroll"),
    min("salary").alias("lowest_salary"),
    max("salary").alias("highest_salary")
).show()

## 6. countDistinct()

In [ ]:
# Count distinct departments
df.agg(
    countDistinct("department").alias("dept_count"),
    countDistinct("location").alias("location_count")
).show()

# Count distinct employees per location
df.groupBy("location").agg(
    countDistinct("emp_id").alias("unique_employees")
).show()

## 7. collect_list() and collect_set()

| Function | Behavior |
|---|---|
| `collect_list` | Aggregates all values into an array (preserves duplicates, order not guaranteed) |
| `collect_set`  | Aggregates unique values into an array (removes duplicates) |

In [ ]:
df.groupBy("department").agg(
    collect_list("name").alias("all_employees"),
    collect_set("location").alias("unique_locations")
).show(truncate=False)

## 8. first() and last()

In [ ]:
# first/last values in each group (order is not guaranteed unless sorted first)
df.groupBy("department").agg(
    first("name").alias("first_employee"),
    last("name").alias("last_employee")
).show()

## 9. pivot() — Reshape Rows to Columns

`pivot()` takes unique values from a column and turns each value into its own column.

In [ ]:
# Total salary by department per year — pivot year into columns
df.groupBy("department").pivot("year").agg(
    round(sum("salary"), 2)
).show()

In [ ]:
# Headcount per department per location — pivot location
df.groupBy("department").pivot("location", ["NYC", "SF"]).agg(
    count("emp_id")
).fillna(0).show()

## 10. rollup() — Hierarchical Subtotals

`rollup(a, b)` produces aggregations at:
- (a, b) level
- (a) level subtotals
- Grand total (null, null)

In [ ]:
from pyspark.sql.functions import when

rollup_df = (
    df.rollup("department", "location")
    .agg(
        count("emp_id").alias("headcount"),
        round(sum("salary"), 2).alias("total_salary")
    )
    .orderBy("department", "location")
)
rollup_df.show()

# Rows where department is null = grand total
# Rows where location is null but department is not = department subtotal

## 11. cube() — All Combinations of Subtotals

`cube(a, b)` produces all grouping combinations:
- (a, b), (a), (b), and grand total

In [ ]:
cube_df = (
    df.cube("department", "location")
    .agg(
        count("emp_id").alias("headcount"),
        round(sum("salary"), 2).alias("total_salary")
    )
    .orderBy("department", "location")
)
cube_df.show()

## 12. Filter After Aggregation (HAVING equivalent)

Use `filter()` on the aggregated result — equivalent to SQL `HAVING`.

In [ ]:
df.groupBy("department").agg(
    count("emp_id").alias("headcount"),
    round(avg("salary"), 2).alias("avg_salary")
).filter(col("headcount") > 2).show()

## 13. Using agg() with Column Expressions

In [ ]:
# You can also use col() expressions inside agg
from pyspark.sql import functions as F

df.groupBy("department").agg(
    F.sum(F.when(col("year") == 2023, col("salary")).otherwise(0)).alias("salary_2023"),
    F.sum(F.when(col("year") == 2022, col("salary")).otherwise(0)).alias("salary_2022"),
).show()

## Key Takeaways

| Pattern | Code |
|---|---|
| Count per group | `groupBy("col").count()` |
| Multiple aggs | `groupBy("col").agg(sum("x"), avg("y"))` |
| Whole DF agg | `df.agg(sum("x"), avg("y"))` |
| Unique counts | `countDistinct("col")` |
| Collect to list | `collect_list("col")` |
| Collect unique | `collect_set("col")` |
| Pivot | `groupBy("a").pivot("b").agg(sum("c"))` |
| Subtotals | `rollup("a","b").agg(...)` |
| All combos | `cube("a","b").agg(...)` |
| HAVING | Chain `.filter()` after `.agg()` |